<table style="border: none" align="center">
   <tr style="border: none">
      <th style="border: none"><font face="verdana" size="6" color="black"><b>使用 ART + ResNet 展示对抗训练（CIFAR-10）</b></font></th>
   </tr>
</table>

## Contents

1. [导入依赖及数据](#prereqs)
2. [构建 ResNet 模型](#resnet)
3. [训练并评估基础 ResNet 分类器](#classifier)
4. [对抗训练鲁棒 ResNet 分类器](#adv_training)
5. [评估与对比](#evaluation)

<a id="prereqs"></a>
## 1. 导入依赖及数据

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.compat.v1.disable_eager_execution()

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation, Add,
    GlobalAveragePooling2D, Dense
)
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.callbacks import LearningRateScheduler

from art.utils import load_dataset
from art.estimators.classification import KerasClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.defences.trainer import AdversarialTrainer

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
(x_train, y_train), (x_test, y_test), min_, max_ = load_dataset('cifar10')

class_names = ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车']

print(f'训练集: {x_train.shape}, 测试集: {x_test.shape}')
print(f'像素范围: [{min_}, {max_}]')

<a id="resnet"></a>
## 2. 构建 ResNet 模型

使用适合 CIFAR-10 的 **ResNet-20**（轻量版，专为 32×32 图像设计）。
核心结构：残差块通过 shortcut 连接缓解梯度消失，比普通 CNN 收敛更快、精度更高。

In [ ]:
def resnet_layer(inputs, filters=16, kernel_size=3, strides=1,
                 activation='relu', batch_norm=True):
    """ResNet 基础卷积层：Conv -> BN -> Activation"""
    x = Conv2D(filters, kernel_size=kernel_size, strides=strides,
               padding='same', kernel_initializer='he_normal',
               use_bias=not batch_norm)(inputs)
    if batch_norm:
        x = BatchNormalization()(x)
    if activation:
        x = Activation(activation)(x)
    return x


def build_resnet(input_shape=(32, 32, 3), num_classes=10, n=3):
    """
    CIFAR-10 专用 ResNet。
    层数 = 6n + 2。默认 n=3 → ResNet-20；n=5 → ResNet-32；n=9 → ResNet-56。
    """
    num_filters = 16
    inputs = Input(shape=input_shape)

    # 第一层卷积
    x = resnet_layer(inputs, filters=num_filters)

    # 3 个 stage，每个 stage 有 n 个残差块
    for stage in range(3):
        for block in range(n):
            strides = 1
            if stage > 0 and block == 0:   # 每个 stage 首块下采样
                strides = 2

            # 残差路径
            y = resnet_layer(x, filters=num_filters, strides=strides)
            y = resnet_layer(y, filters=num_filters, activation=None)

            # Shortcut 连接：维度不同时用 1×1 卷积对齐
            if strides > 1:
                x = resnet_layer(x, filters=num_filters, kernel_size=1,
                                 strides=strides, activation=None,
                                 batch_norm=False)

            x = Add()([x, y])
            x = Activation('relu')(x)

        num_filters *= 2   # 每个 stage 后通道数翻倍：16 → 32 → 64

    # 输出头
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation='softmax',
                    kernel_initializer='he_normal')(x)

    return Model(inputs=inputs, outputs=outputs)


# 学习率衰减策略
def lr_schedule(epoch):
    lr = 1e-3
    if epoch >= 80:  lr = 1e-5
    elif epoch >= 60: lr = 1e-4
    elif epoch >= 40: lr = 5e-4
    return lr


print('ResNet 模块定义完毕')

<a id="classifier"></a>
## 3. 训练并评估基础 ResNet 分类器

In [ ]:
# 构建 ResNet-20（n=3）
resnet_model = build_resnet(input_shape=(32, 32, 3), num_classes=10, n=3)
resnet_model.compile(
    loss=categorical_crossentropy,
    optimizer=Adam(learning_rate=lr_schedule(0)),
    metrics=['accuracy']
)
resnet_model.summary()

In [ ]:
# 封装为 ART 分类器
classifier = KerasClassifier(
    clip_values=(min_, max_),
    model=resnet_model,
    use_logits=False
)

# 训练（建议 100 epochs；资源有限可先跑 30 epochs 验证流程）
lr_callback = LearningRateScheduler(lr_schedule)
classifier.fit(
    x_train, y_train,
    nb_epochs=100,
    batch_size=128,
    callbacks=[lr_callback]
)

# resnet_model.save('./cifar10_resnet20_original.h5')

In [ ]:
# 评估基础 ResNet 在原始测试集上的准确率
x_test_pred = np.argmax(classifier.predict(x_test), axis=1)
nb_correct_pred = np.sum(x_test_pred == np.argmax(y_test, axis=1))

print('=== 基础 ResNet — 原始测试集 ===')
print(f'正确分类: {nb_correct_pred} / {len(x_test)}')
print(f'准确率:   {nb_correct_pred / len(x_test) * 100:.2f}%')

In [ ]:
# FGM 攻击：生成对抗样本
attacker = FastGradientMethod(classifier, eps=0.05)
x_test_adv = attacker.generate(x_test[:200], y_test[:200])

# ── 可视化：原始 / 对抗 / 扰动（×10 放大）三行对比 ──
n_show = 5
fig, axes = plt.subplots(3, n_show, figsize=(13, 7))
row_titles = ['原始图像', '对抗图像\n(eps=0.05)', '扰动×10\n(放大显示)']

for i in range(n_show):
    orig  = x_test[i].clip(0, 1)
    adv   = x_test_adv[i].clip(0, 1)
    noise = (x_test_adv[i] - x_test[i]) * 10 + 0.5  # 扰动放大，居中到 0.5

    true_label = class_names[np.argmax(y_test[i])]
    pred_orig  = class_names[np.argmax(classifier.predict(x_test[i:i+1])[0])]
    pred_adv   = class_names[np.argmax(classifier.predict(x_test_adv[i:i+1])[0])]

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'真实: {true_label}\n预测: {pred_orig}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(adv)
    axes[1, i].set_title(f'预测: {pred_adv}', fontsize=9,
                         color='red' if pred_adv != true_label else 'green')
    axes[1, i].axis('off')

    axes[2, i].imshow(noise.clip(0, 1))
    axes[2, i].set_title(f'max Δ={np.max(np.abs(x_test_adv[i]-x_test[i])):.3f}', fontsize=9)
    axes[2, i].axis('off')

# 行标签
for row, title in enumerate(row_titles):
    axes[row, 0].set_ylabel(title, fontsize=10, rotation=0,
                             labelpad=55, va='center')

plt.suptitle('基础 ResNet-20：FGM 对抗样本可视化\n（红色标题=预测错误，绿色=预测正确）',
             fontsize=12)
plt.tight_layout()
plt.show()

# 准确率统计
x_test_adv_pred = np.argmax(classifier.predict(x_test_adv), axis=1)
nb_correct_adv  = np.sum(x_test_adv_pred == np.argmax(y_test[:200], axis=1))
print(f'基础 ResNet — FGM 对抗样本准确率: {nb_correct_adv / 200 * 100:.2f}%')


<a id="adv_training"></a>
## 4. 对抗训练鲁棒 ResNet 分类器

使用 **PGD 对抗训练**（Madry et al., 2018）：每个训练 batch 先用 PGD 生成对抗样本再更新参数。

In [ ]:
# 构建结构相同的鲁棒 ResNet（独立模型，避免污染基础模型权重）
robust_resnet_model = build_resnet(input_shape=(32, 32, 3), num_classes=10, n=3)
robust_resnet_model.compile(
    loss=categorical_crossentropy,
    optimizer=Adam(learning_rate=lr_schedule(0)),
    metrics=['accuracy']
)

robust_classifier = KerasClassifier(
    clip_values=(min_, max_),
    model=robust_resnet_model,
    use_logits=False
)

# PGD 攻击：eps=8/255≈0.031，训练期间作为数据增强使用
pgd_attack = ProjectedGradientDescent(
    estimator=robust_classifier,
    eps=0.031,
    eps_step=0.007,
    max_iter=10,       # 训练时迭代步数不宜过多，以控制速度
    verbose=False
)

print('鲁棒 ResNet 与 PGD 攻击初始化完毕')

In [ ]:
# 对抗训练（在 GPU 上约需 1-2 小时，CPU 上更长）
# ratio=1.0：每个 batch 全部替换为对抗样本（最严格）
# ratio=0.5：一半干净样本 + 一半对抗样本（速度更快，精度损失更小）
trainer = AdversarialTrainer(robust_classifier, pgd_attack, ratio=0.5)
trainer.fit(
    x_train, y_train,
    nb_epochs=100,
    batch_size=128,
    callbacks=[LearningRateScheduler(lr_schedule)]
)

# robust_resnet_model.save('./cifar10_resnet20_robust.h5')

<a id="evaluation"></a>
## 5. 评估与对比

In [ ]:
# ── 原始测试集准确率对比 ──
x_test_robust_pred = np.argmax(robust_classifier.predict(x_test), axis=1)
nb_correct_robust  = np.sum(x_test_robust_pred == np.argmax(y_test, axis=1))

print('=' * 45)
print('           原始测试集准确率（干净样本）')
print('=' * 45)
print(f'  基础 ResNet-20 :  {nb_correct_pred / len(x_test) * 100:.2f}%')
print(f'  鲁棒 ResNet-20 :  {nb_correct_robust / len(x_test) * 100:.2f}%')
print()

# ── 对抗样本准确率对比（FGM + PGD 两种攻击）──
nb_eval = 500   # 用 500 个样本评估，兼顾速度与代表性

# FGM 攻击
fgm_orig = FastGradientMethod(classifier,        eps=0.05)
fgm_rob  = FastGradientMethod(robust_classifier, eps=0.05)
adv_fgm_orig = fgm_orig.generate(x_test[:nb_eval], y_test[:nb_eval])
adv_fgm_rob  = fgm_rob.generate( x_test[:nb_eval], y_test[:nb_eval])

labels_eval = np.argmax(y_test[:nb_eval], axis=1)
acc_fgm_orig = np.mean(np.argmax(classifier.predict(adv_fgm_orig),        axis=1) == labels_eval)
acc_fgm_rob  = np.mean(np.argmax(robust_classifier.predict(adv_fgm_rob),  axis=1) == labels_eval)

# PGD 攻击
pgd_orig = ProjectedGradientDescent(estimator=classifier,        eps=0.031, eps_step=0.007, max_iter=20, verbose=False)
pgd_rob  = ProjectedGradientDescent(estimator=robust_classifier, eps=0.031, eps_step=0.007, max_iter=20, verbose=False)
adv_pgd_orig = pgd_orig.generate(x_test[:nb_eval], y_test[:nb_eval])
adv_pgd_rob  = pgd_rob.generate( x_test[:nb_eval], y_test[:nb_eval])

acc_pgd_orig = np.mean(np.argmax(classifier.predict(adv_pgd_orig),        axis=1) == labels_eval)
acc_pgd_rob  = np.mean(np.argmax(robust_classifier.predict(adv_pgd_rob),  axis=1) == labels_eval)

print('=' * 45)
print('        对抗样本准确率对比（白盒攻击）')
print('=' * 45)
print(f'  攻击方式       基础 ResNet   鲁棒 ResNet')
print(f'  FGM(eps=0.05)  {acc_fgm_orig*100:6.2f}%      {acc_fgm_rob*100:6.2f}%')
print(f'  PGD(eps=0.031) {acc_pgd_orig*100:6.2f}%      {acc_pgd_rob*100:6.2f}%')
print('=' * 45)
print('（鲁棒模型通过对抗训练，在对抗样本上准确率应显著更高）')


In [ ]:
# ── 在一系列 eps 下用 PGD 白盒攻击对比两个模型 ──
eps_range = [0.0, 0.005, 0.01, 0.02, 0.031, 0.05, 0.08, 0.1]
acc_original  = []
acc_robust    = []
nb_samples    = 200   # 取前 200 个样本，速度与精度的折中

# eps=0 时直接用干净样本
pred_clean_orig = np.argmax(classifier.predict(x_test[:nb_samples]), axis=1)
pred_clean_rob  = np.argmax(robust_classifier.predict(x_test[:nb_samples]), axis=1)
labels          = np.argmax(y_test[:nb_samples], axis=1)
acc_original.append(np.mean(pred_clean_orig == labels))
acc_robust.append(np.mean(pred_clean_rob  == labels))

pgd_eval_orig = ProjectedGradientDescent(
    estimator=classifier, eps=0.01, eps_step=0.003, max_iter=20, verbose=False)
pgd_eval_rob  = ProjectedGradientDescent(
    estimator=robust_classifier, eps=0.01, eps_step=0.003, max_iter=20, verbose=False)

for eps in eps_range[1:]:
    pgd_eval_orig.set_params(eps=eps, eps_step=eps/5)
    pgd_eval_rob.set_params( eps=eps, eps_step=eps/5)

    adv_orig = pgd_eval_orig.generate(x_test[:nb_samples], y_test[:nb_samples])
    adv_rob  = pgd_eval_rob.generate( x_test[:nb_samples], y_test[:nb_samples])

    acc_original.append(np.mean(
        np.argmax(classifier.predict(adv_orig), axis=1) == labels))
    acc_robust.append(np.mean(
        np.argmax(robust_classifier.predict(adv_rob), axis=1) == labels))

print('评估完成')

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(eps_range, acc_original, 'b-o', label='基础 ResNet-20')
ax.plot(eps_range, acc_robust,   'r-o', label='对抗训练 ResNet-20')

ax.set_xlabel('扰动大小（eps，L∞ 范数）')
ax.set_ylabel('分类准确率')
ax.set_title('CIFAR-10：ResNet-20 基础模型 vs 对抗训练模型（PGD 白盒攻击）')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('cifar10_resnet_robustness.jpg', dpi=200, bbox_inches='tight')
plt.show()